# 🎬 DeepfakeDetect — Video Frame Training (Colab)

Fine-tunes **EfficientNet-B4** on extracted video frames for deepfake detection.

### What this notebook does
1. Installs dependencies
2. Uploads your project files + video frame dataset
3. Trains with video-specific augmentations + temporal consistency loss
4. Evaluates with **per-video AUC** (the realistic metric — not per-frame)
5. Downloads the trained checkpoint for use with the FastAPI server

### Before you start
- **Runtime → Change runtime type → T4 GPU** (free tier works fine)
- Prepare your frame dataset using `extract_video_frames.py` locally,
  then zip the `video_frames/` folder and upload it here.
- OR use one of the public dataset options in Step 3.

### Key concepts
| Concept | Why it matters |
|---------|---------------|
| Video-level split | Prevents data leakage — adjacent frames from the same video are never in both train and val |
| Video augmentations | JPEG compression, motion blur, block artifacts — makes model robust to re-encoding |
| Temporal consistency loss | Penalises inconsistent predictions on frames from the same video |
| Video-level AUC | Aggregates per-frame scores per video before computing AUC — the realistic deployment metric |

## Step 1 — Check GPU & Install Dependencies

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
!pip install -q timm scikit-learn tqdm matplotlib opencv-python-headless Pillow

## Step 2 — Upload Project Files

In [ ]:
from google.colab import files
import zipfile, os, shutil

print('Upload train_video.py and extract_video_frames.py from your project folder')
uploaded = files.upload()

## Step 3 — Prepare Dataset

**Option A — Upload your own frame dataset (recommended)**  
Run `extract_video_frames.py` on your local machine first to extract frames from your videos,
then zip the `video_frames/` folder and upload the zip here.

**Option B — Use FaceForensics++ (requires academic license)**  
Request access at https://github.com/ondyari/FaceForensics

**Option C — Use CelebDF-v2 (free)**  
Download from https://github.com/yuezunli/celeb-deepfakeforensics

**Option D — Quick demo with synthetic data (no real dataset)**  
The cell below creates a tiny synthetic dataset for testing the pipeline.

In [ ]:
# ── Option A: Upload your own frames zip ──────────────────────────────
# Uncomment and run this cell if you have your own frame dataset

# from google.colab import files
# print('Upload your video_frames.zip')
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
# import zipfile
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall('.')
# print('Extracted. Checking structure ...')
# !find video_frames -maxdepth 3 | head -20

In [ ]:
# ── Option D: Synthetic demo dataset (no real videos needed) ──────────
# Creates small random-image folders in the expected structure.
# Replace with real data for meaningful training.

import os
import numpy as np
from PIL import Image

def make_synthetic_dataset(root='video_frames', n_per_class=200):
    for split in ['train', 'val']:
        for cls in ['real', 'fake']:
            d = os.path.join(root, split, cls)
            os.makedirs(d, exist_ok=True)
            n = n_per_class if split == 'train' else n_per_class // 5
            for i in range(n):
                # Fake frames: slightly different pixel distribution
                if cls == 'fake':
                    arr = np.random.randint(100, 200, (224, 224, 3), dtype=np.uint8)
                else:
                    arr = np.random.randint(30, 150, (224, 224, 3), dtype=np.uint8)
                # Encode video_id in filename (vid001_000000.jpg)
                vid_id = f'vid{i//10:03d}'
                fname = f'{vid_id}_{i:06d}.jpg'
                Image.fromarray(arr).save(os.path.join(d, fname))
    print(f'✅ Synthetic dataset created at {root}/')
    for split in ['train', 'val']:
        for cls in ['real', 'fake']:
            p = os.path.join(root, split, cls)
            print(f'  {split}/{cls}: {len(os.listdir(p))} frames')

make_synthetic_dataset(n_per_class=300)

## Step 4 — Configure Training

In [ ]:
# ── Training configuration ────────────────────────────────────────────
CONFIG = {
    'data_dir':          'video_frames',   # path to your frame dataset
    'model_dir':         'models',         # where to save checkpoints
    'model_name':        'efficientnet_b4',
    'img_size':          224,
    'epochs':            15,
    'batch_size':        32,               # reduce to 16 if OOM
    'lr':                2e-4,
    'dropout':           0.4,
    'tc_loss_weight':    0.1,              # temporal consistency loss weight
    'freeze_epochs':     2,                # warm-up with frozen backbone
    'balance_sampler':   True,             # balance real/fake frames per batch
    'pos_weight':        1.0,              # increase if fake frames are rare
    'num_workers':       2,
}
print('Config ready:', CONFIG)

## Step 5 — Train

In [ ]:
# Build the argument string and run train_video.py
import subprocess, sys

cmd = [
    sys.executable, 'train_video.py',
    '--data_dir',         CONFIG['data_dir'],
    '--model_dir',        CONFIG['model_dir'],
    '--model_name',       CONFIG['model_name'],
    '--img_size',         str(CONFIG['img_size']),
    '--epochs',           str(CONFIG['epochs']),
    '--batch_size',       str(CONFIG['batch_size']),
    '--lr',               str(CONFIG['lr']),
    '--dropout',          str(CONFIG['dropout']),
    '--tc_loss_weight',   str(CONFIG['tc_loss_weight']),
    '--freeze_epochs',    str(CONFIG['freeze_epochs']),
    '--pos_weight',       str(CONFIG['pos_weight']),
    '--num_workers',      str(CONFIG['num_workers']),
]
if CONFIG['balance_sampler']:
    cmd.append('--balance_sampler')

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('Return code:', result.returncode)

## Step 6 — View Results

In [ ]:
from IPython.display import Image as IPImage
import os

curve_path = os.path.join(CONFIG['model_dir'], 'video_training_curves.png')
if os.path.exists(curve_path):
    display(IPImage(curve_path))
else:
    print('Training curves not found — training may not have completed.')

In [ ]:
# Quick sanity check on the saved checkpoint
import torch, os

ckpt_path = os.path.join(CONFIG['model_dir'], 'best_video_model.pth')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print('✅ Checkpoint loaded successfully')
    print(f'   Model:      {ckpt["model_name"]}')
    print(f'   Img size:   {ckpt["img_size"]}')
    print(f'   Val AUC:    {ckpt["val_auc"]:.4f}')
    print(f'   Val Acc:    {ckpt["val_acc"]:.4f}')
    print(f'   Trained on: {ckpt.get("trained_on", "unknown")}')
    print(f'   File size:  {os.path.getsize(ckpt_path) / 1e6:.1f} MB')
else:
    print('⚠️  Checkpoint not found. Check training output above for errors.')

## Step 7 — Download Checkpoint

In [ ]:
from google.colab import files
import os

ckpt_path = os.path.join(CONFIG['model_dir'], 'best_video_model.pth')
curves_path = os.path.join(CONFIG['model_dir'], 'video_training_curves.png')

if os.path.exists(ckpt_path):
    files.download(ckpt_path)
    print('📥 Downloading checkpoint ...')
else:
    print('⚠️  No checkpoint found.')

if os.path.exists(curves_path):
    files.download(curves_path)
    print('📥 Downloading training curves ...')

## Step 8 — Use Your Model

Place `best_video_model.pth` in your project's `models/` folder, then start the API with:

```bash
MODEL_PATH=./models/best_video_model.pth uvicorn main:app --port 8000
```

The video detection endpoint (`/predict/video`) will automatically use your fine-tuned model
instead of the HuggingFace ensemble. Image detection (`/predict`) continues to use the ensemble.

---

### Improving accuracy tips

| Tip | How |
|-----|-----|
| More data | Aim for 1000+ videos per class. FaceForensics++ has 1000 real + 5000 fake |
| Face crop | Run `extract_video_frames.py --face_crop` — focuses model on face region |
| Larger model | Try `--model_name efficientnet_b7` or `convnext_base` for better accuracy at cost of speed |
| Higher resolution | `--img_size 299` or `384` if VRAM allows |
| More epochs | `--epochs 30` with early stopping based on video-level AUC |
| Mixed datasets | Combine FaceForensics++ + CelebDF + DFDC for broader coverage |